# Importancia de la cláusula — verificación numérica

Regenera todas las cifras de `docs/metricas/importancia-clausula.md`. Llama al módulo que
ejecuta el servidor, `server/services/graph/knowledge/personalized_pagerank.py`, no a una
réplica.

La importancia de una cláusula es **cuánta masa le llega desde los paseos que empiezan en
las demás cláusulas**:

$$\pi_c(v_i)=\begin{cases}\dfrac{1}{|V_c|} & v_i\in V_c\\[4pt] 0 & \text{en otro caso}\end{cases}$$

$$PR_{c,i}^{(t+1)}=\bigl(1-d+dD_c^{(t)}\bigr)\,\pi_c(v_i)+d\sum_{j\in\mathcal{N}_i}\frac{PR_{c,j}^{(t)}}{\deg(v_j)}$$

$$I(c)=\frac{1}{|C|}\sum_{\substack{c'\in C\\ c'\neq c}}\;\sum_{v_i\in V_c}PR_{c',i}$$

El paseo recorre solo cláusulas, obligaciones, derechos y prohibiciones.

Se cuentan los mismos enunciados que cuenta la tabla: los que tienen cláusula y pertenecen a
una de las partes (obligaciones y prohibiciones con la parte a la que obligan, derechos con
su titular).

In [1]:
import json
import sys
from pathlib import Path

import networkx as nx
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'infra').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from server.services.graph.knowledge import personalized_pagerank as ppr

KG_DIR = ROOT / 'infra' / 'json' / 'kg'
RELIABLE = ('BELLICUM', 'SteelVault')  # los grafos fiables que hay en infra/json/kg


def short(path):
    return path.stem.removeprefix('root_')[:24]


def owner(statement, collection):
    return statement.get('benefitPartyId') if collection == 'rights' else statement.get('burdenPartyId')


def table_rows(kg):
    # Cláusula -> enunciados que la tabla muestra en las columnas de las dos partes.
    parties = {p['id'] for p in kg['parties']}
    rows = {}
    for collection in ppr.DEONTIC_COLLECTIONS:
        for statement in kg[collection]:
            if statement.get('clauseId') and owner(statement, collection) in parties:
                rows.setdefault(statement['clauseId'], []).append(statement['id'])
    return rows


def headings(kg):
    return {c['id']: ' '.join(x for x in (c.get('ref'), c.get('heading')) if x) or c['id'] for c in kg['clauses']}


def ranks(scores):
    # Puestos con empates promediados.
    keys = sorted(scores, key=lambda k: -scores[k])
    out, i = {}, 0
    while i < len(keys):
        j = i
        while j + 1 < len(keys) and scores[keys[j + 1]] == scores[keys[i]]:
            j += 1
        for k in keys[i:j + 1]:
            out[k] = (i + j) / 2
        i = j + 1
    return out


def spearman(a, b):
    ra, rb = ranks(a), ranks({k: b[k] for k in a})
    x, y = np.array([ra[k] for k in a]), np.array([rb[k] for k in a])
    if x.std() == 0 or y.std() == 0:
        return float('nan')
    return float(np.corrcoef(x, y)[0, 1])


def top(scores, k=10):
    return set(sorted(scores, key=lambda c: -scores[c])[:k])


def settled(scores, noise=1e-9):
    # networkx converge hasta ~1e-13 por nodo: por debajo de `noise` es cero, y se redondea
    # para que ese resto no rompa empates.
    return {k: (round(v, 11) if v > noise else 0.0) for k, v in scores.items()}


contracts = []
for path in sorted(KG_DIR.glob('*.json')):
    kg = json.loads(path.read_text(encoding='utf-8'))
    rows = table_rows(kg)
    if len(rows) >= 3:
        contracts.append((path, kg, rows))
[short(p) for p, _, _ in contracts]

['BELLICUMPHARMACEUTICALS_',
 'CARTAO_DE_CREDITO_SANTAN',
 'SteelVaultCorp_20081224_',
 'Synthetic_HalversonCorvi',
 'Synthetic_OrvaneTalvera_',
 'lease_contract_10',
 'lease_contract_166',
 'lease_contract_201']

## La matriz de paseos

`landed[c, c']` es la fracción de la masa del paseo que empieza en $c'$ que termina en los
enunciados de $c$, es decir, $\sum_{v_i\in V_c}PR_{c',i}$. Se calcula con `networkx`, de forma
independiente del módulo, sobre el grafo de cláusulas y enunciados.

- Cada **columna** de `landed` es un paseo.
- Cada **fila** es una cláusula que recibe.
- La importancia es la suma de la fila **sin la diagonal**, dividida por $|C|$.

In [2]:
def walk_graph(kg, collections=ppr.INFLOW_COLLECTIONS):
    keep = {n['id'] for c in collections for n in kg.get(c) or []}
    graph = nx.MultiGraph()
    graph.add_nodes_from(keep)
    for edge in kg['edges']:
        if edge['source'] in keep and edge['target'] in keep:
            graph.add_edge(edge['source'], edge['target'])
    return graph


def landed_matrix(kg, rows, damping=ppr.DAMPING, collections=ppr.INFLOW_COLLECTIONS):
    graph = walk_graph(kg, collections)
    clauses = list(rows)
    landed = np.zeros((len(clauses), len(clauses)))
    for j, start in enumerate(clauses):
        seed = {n: 0.0 for n in graph}
        for statement in rows[start]:
            seed[statement] = 1 / len(rows[start])
        rank = nx.pagerank(graph, alpha=damping, personalization=seed, dangling=seed, tol=1e-13, max_iter=2000)
        for i, clause in enumerate(clauses):
            landed[i, j] = sum(rank[s] for s in rows[clause])
    return clauses, landed


matrices = {path: landed_matrix(kg, rows) for path, kg, rows in contracts}

## 1. El módulo calcula lo que dice la fórmula

In [3]:
for path, kg, rows in contracts:
    clauses, landed = matrices[path]
    reference = (landed.sum(axis=1) - np.diag(landed)) / len(clauses)
    module = ppr.compute(kg, [s for g in rows.values() for s in g]).by_clause
    gap = max(abs(module[c] - reference[i]) for i, c in enumerate(clauses))
    print(f'{short(path):26} filas {len(clauses):>3} · diferencia con networkx {gap:.0e}')

BELLICUMPHARMACEUTICALS_   filas  91 · diferencia con networkx 2e-12
CARTAO_DE_CREDITO_SANTAN   filas  81 · diferencia con networkx 2e-12
SteelVaultCorp_20081224_   filas  22 · diferencia con networkx 2e-11
Synthetic_HalversonCorvi   filas  19 · diferencia con networkx 2e-11
Synthetic_OrvaneTalvera_   filas  48 · diferencia con networkx 3e-12
lease_contract_10          filas  20 · diferencia con networkx 6e-12
lease_contract_166         filas   9 · diferencia con networkx 3e-13
lease_contract_201         filas   8 · diferencia con networkx 3e-11


## 2. Sesgo por tamaño: método anterior frente al actual

El **método anterior** es un solo paseo sobre el grafo entero (partes, términos, condiciones,
referencias y valores incluidos), con el prior $1/(|C|\cdot|V_c|)$, sumando lo que retienen los
enunciados de cada fila. El módulo lo sigue calculando para la pestaña del grafo: es la suma
de `by_statement`.

ρ es la correlación de Spearman entre el orden de las filas y su número de enunciados:
cerca de 1, el orden solo cuenta enunciados; cerca de 0, el tamaño no lo decide.

In [4]:
for path, kg, rows in contracts:
    result = ppr.compute(kg, [s for g in rows.values() for s in g])
    previous = {c: sum(result.by_statement[s] for s in g) for c, g in rows.items()}
    current = result.by_clause
    size = {c: len(g) for c, g in rows.items()}
    zeros = sum(1 for v in current.values() if v == 0)
    rho_now = f'{spearman(current, size):.2f}' if zeros < len(rows) else '—'
    print(f'{short(path):26} ρ anterior {spearman(previous, size):>5.2f} · ρ actual {rho_now:>5} · '
          f'sin conexión {zeros:>2} de {len(rows)}')

BELLICUMPHARMACEUTICALS_   ρ anterior  0.92 · ρ actual  0.48 · sin conexión  0 de 91
CARTAO_DE_CREDITO_SANTAN   ρ anterior  0.85 · ρ actual  0.50 · sin conexión  3 de 81
SteelVaultCorp_20081224_   ρ anterior  0.83 · ρ actual -0.22 · sin conexión  7 de 22
Synthetic_HalversonCorvi   ρ anterior  0.89 · ρ actual -0.22 · sin conexión  9 de 19
Synthetic_OrvaneTalvera_   ρ anterior  0.59 · ρ actual -0.03 · sin conexión 26 de 48
lease_contract_10          ρ anterior  0.82 · ρ actual -0.12 · sin conexión  8 de 20
lease_contract_166         ρ anterior  0.81 · ρ actual     — · sin conexión  9 de 9
lease_contract_201         ρ anterior  0.89 · ρ actual  0.04 · sin conexión  2 de 8


### Con las partes dentro del paseo

La misma importancia (entrada desde las demás cláusulas), pero con los nodos de parte en el
grafo del paseo. Cada enunciado está unido a su parte, y la parte reparte lo que recibe entre
todos sus enunciados.

In [5]:
for path, kg, rows in contracts:
    clauses, landed = landed_matrix(kg, rows, collections=('parties', *ppr.INFLOW_COLLECTIONS))
    incoming = settled({c: (landed[i].sum() - landed[i, i]) / len(clauses) for i, c in enumerate(clauses)})
    size = {c: len(rows[c]) for c in clauses}
    print(f'{short(path):26} ρ con las partes en el paseo {spearman(incoming, size):>5.2f}')

BELLICUMPHARMACEUTICALS_   ρ con las partes en el paseo  0.97


CARTAO_DE_CREDITO_SANTAN   ρ con las partes en el paseo  0.93
SteelVaultCorp_20081224_   ρ con las partes en el paseo  0.94


Synthetic_HalversonCorvi   ρ con las partes en el paseo  0.94


Synthetic_OrvaneTalvera_   ρ con las partes en el paseo  0.85
lease_contract_10          ρ con las partes en el paseo  0.97
lease_contract_166         ρ con las partes en el paseo  0.97
lease_contract_201         ρ con las partes en el paseo  0.89


## 3. Por qué un paseo por cláusula y no uno solo

Con un solo paseo, sembrado con $1/(|C|\cdot|V_c|)$ sobre el mismo grafo de cláusulas y
enunciados, el valor de cada fila es la suma de su fila entera de `landed`, diagonal
incluida (por linealidad). La diagonal es la masa que la cláusula se queda a sí misma.
Esta celda mide qué parte del valor es esa masa propia.

In [6]:
for path, kg, rows in contracts:
    clauses, landed = matrices[path]
    single = landed.sum(axis=1) / len(clauses)
    own = np.diag(landed) / landed.sum(axis=1)
    size = {c: len(rows[c]) for c in clauses}
    rho_single = spearman({c: single[i] for i, c in enumerate(clauses)}, size)
    print(f'{short(path):26} masa propia: media {own.mean():>4.0%}, de {own.min():>4.0%} a {own.max():>4.0%} · ρ un solo paseo {rho_single:>5.2f}')

BELLICUMPHARMACEUTICALS_   masa propia: media  72%, de  40% a  97% · ρ un solo paseo  0.86
CARTAO_DE_CREDITO_SANTAN   masa propia: media  69%, de  29% a 100% · ρ un solo paseo  0.81
SteelVaultCorp_20081224_   masa propia: media  82%, de  53% a 100% · ρ un solo paseo  0.75
Synthetic_HalversonCorvi   masa propia: media  87%, de  44% a 100% · ρ un solo paseo  0.57
Synthetic_OrvaneTalvera_   masa propia: media  83%, de  32% a 100% · ρ un solo paseo  0.40
lease_contract_10          masa propia: media  86%, de  57% a 100% · ρ un solo paseo  0.89
lease_contract_166         masa propia: media 100%, de 100% a 100% · ρ un solo paseo  0.57
lease_contract_201         masa propia: media  82%, de  50% a 100% · ρ un solo paseo  0.52


## 4. ¿Depende el orden de $d$?

Se repite el cálculo con otros valores de $d$ y se compara con el orden de $d = 0.85$.

In [7]:
for path, kg, rows in contracts:
    base = ppr._inflow(kg, rows)
    if not any(v > 0 for v in base.values()):
        print(f'{short(path):26} sin conexiones: no hay orden que comparar')
        continue
    k = min(10, len(rows))
    cells = []
    for damping in (0.5, 0.7, 0.9, 0.95):
        saved, ppr.DAMPING = ppr.DAMPING, damping
        try:
            other = ppr._inflow(kg, rows)
        finally:
            ppr.DAMPING = saved
        cells.append((spearman(other, base), len(top(other, k) & top(base, k))))
    rho = [c[0] for c in cells]
    kept = [c[1] for c in cells]
    print(f'{short(path):26} ρ {min(rho):.2f} – {max(rho):.2f} · top {k} que se mantiene {min(kept)} – {max(kept)}')

BELLICUMPHARMACEUTICALS_   ρ 0.94 – 0.99 · top 10 que se mantiene 8 – 9
CARTAO_DE_CREDITO_SANTAN   ρ 0.89 – 0.99 · top 10 que se mantiene 7 – 9
SteelVaultCorp_20081224_   ρ 0.91 – 1.00 · top 10 que se mantiene 8 – 10
Synthetic_HalversonCorvi   ρ 0.97 – 1.00 · top 10 que se mantiene 10 – 10
Synthetic_OrvaneTalvera_   ρ 0.99 – 1.00 · top 10 que se mantiene 8 – 9
lease_contract_10          ρ 0.97 – 0.99 · top 10 que se mantiene 9 – 10


lease_contract_166         sin conexiones: no hay orden que comparar
lease_contract_201         ρ 0.93 – 1.00 · top 8 que se mantiene 8 – 8


## 5. Ejemplo: SteelVault

El contrato que usa el documento para explicar la matriz.

In [8]:
path, kg, rows = next(c for c in contracts if 'SteelVault' in c[0].name)
names = headings(kg)
counts = {c: len(kg.get(c) or []) for c in ppr.NODE_COLLECTIONS}
edges = {}
for edge in kg['edges']:
    edges[edge['type']] = edges.get(edge['type'], 0) + 1
print('nodos:', counts, '· total', sum(counts.values()))
print('aristas:', dict(sorted(edges.items(), key=lambda kv: -kv[1])), '· total', sum(edges.values()))

result = ppr.compute(kg, [s for g in rows.values() for s in g])
previous = {c: sum(result.by_statement[s] for s in g) for c, g in rows.items()}
current = result.by_clause
parties = [p['id'] for p in kg['parties']]
print(f"masa en los nodos de parte, método anterior: {sum(result.by_node[p] for p in parties):.1%}")

order = lambda d: {c: i + 1 for i, c in enumerate(sorted(d, key=lambda c: -d[c]))}
before, now = order(previous), order(current)
peak_before, peak_now = max(previous.values()), max(current.values())
print(f'\n|C| = {len(rows)} · prior del método anterior por fila = 1/{len(rows)}')
print(f"{'cláusula':28} {'enunc.':>6} {'antes':>7} {'puesto':>6} {'ahora':>7} {'puesto':>6}")
for c in sorted(rows, key=lambda c: (current[c] == 0, -current[c], before[c])):
    place = now[c] if current[c] > 0 else '–'
    print(f'{names[c][:28]:28} {len(rows[c]):>6} {previous[c] / peak_before:>7.0%} {before[c]:>6} '
          f'{current[c] / peak_now:>7.0%} {place:>6}')

nodos: {'parties': 2, 'clauses': 26, 'definedTerms': 10, 'obligations': 52, 'rights': 19, 'prohibitions': 19, 'conditions': 32, 'references': 10, 'values': 16} · total 186
aristas: {'is_part_of': 159, 'assigns_obligation_to': 62, 'uses': 22, 'grants_right_to': 19, 'defines': 8, 'modifies': 8} · total 278
masa en los nodos de parte, método anterior: 15.5%

|C| = 22 · prior del método anterior por fila = 1/22
cláusula                     enunc.   antes puesto   ahora puesto
11 Liability                      4     36%      4    100%      1
7.2                               5     44%      3     63%      2
5 Compliance                      2     18%     20     42%      3
8.4                               2     24%     12     33%      4
8.1                               2     21%     15     29%      5
5.3                               2     24%     13     27%      8
5.1                               2     20%     16     27%      6
5.5                               2     20%     17     27%   

La matriz de paseos para cuatro de sus cláusulas, en porcentaje de la masa de cada paseo,
y de dónde le llega a *Compensation* su valor en el método anterior.

In [9]:
clauses, landed = matrices[path]
pick = ['11 Liability', '8.4', '9 Indemnification', '1 Compensation']
idx = [next(i for i, c in enumerate(clauses) if names[c] == name) for name in pick]
print('el paseo empieza en ↓ / deja masa en →'.ljust(40) + ''.join(f'{n[:14]:>16}' for n in pick))
for j in idx:
    print(f'{names[clauses[j]][:38]:40}' + ''.join(f'{landed[i, j]:>16.1%}' for i in idx))

for name in pick:
    i = idx[pick.index(name)]
    incoming = (landed[i].sum() - landed[i, i]) / len(clauses)
    print(f'\n{name}: entrada = {incoming:.4f}; masa propia (no cuenta) = {landed[i, i]:.1%} de su propio paseo')
    givers = sorted(((landed[i, j], names[clauses[j]]) for j in range(len(clauses)) if j != i and landed[i, j] > 1e-9), reverse=True)
    print('   le llega de:', ', '.join(f'{n} {v:.1%}' for v, n in givers[:6]) or 'ninguna cláusula')

# Método anterior: qué parte del valor de cada cláusula llega de otras cláusulas, con el grafo entero.
full = {n['id'] for c in ppr.NODE_COLLECTIONS for n in kg.get(c) or []}
graph = nx.MultiGraph()
graph.add_nodes_from(full)
for edge in kg['edges']:
    if edge['source'] in full and edge['target'] in full:
        graph.add_edge(edge['source'], edge['target'])
compensation = next(c for c in clauses if names[c] == '1 Compensation')
own_part, total = 0.0, 0.0
for start in clauses:
    seed = {n: 0.0 for n in graph}
    for s in rows[start]:
        seed[s] = 1 / len(rows[start])
    rank = nx.pagerank(graph, alpha=ppr.DAMPING, personalization=seed, dangling=seed, tol=1e-13, max_iter=2000)
    mass = sum(rank[s] for s in rows[compensation])
    total += mass
    own_part += mass if start == compensation else 0.0
print(f'\nMétodo anterior: el {1 - own_part / total:.0%} del valor de Compensation llega de otras cláusulas')

el paseo empieza en ↓ / deja masa en →      11 Liability             8.4  9 Indemnificat  1 Compensation
11 Liability                                       40.7%            3.5%            5.5%            0.0%
8.4                                                11.3%           26.9%            0.8%            0.0%
9 Indemnification                                  18.0%            0.8%           34.4%            0.0%
1 Compensation                                      0.0%            0.0%            0.0%           52.1%

11 Liability: entrada = 0.0168; masa propia (no cuenta) = 40.7% de su propio paseo
   le llega de: 9 Indemnification 18.0%, 8.4 11.3%, 8.2 2.7%, 8.3 2.7%, 8.1 2.2%

8.4: entrada = 0.0055; masa propia (no cuenta) = 26.9% de su propio paseo
   le llega de: 11 Liability 3.5%, 8.3 2.8%, 8.2 2.8%, 8.1 2.3%, 9 Indemnification 0.8%

9 Indemnification: entrada = 0.0031; masa propia (no cuenta) = 34.4% de su propio paseo
   le llega de: 11 Liability 5.5%, 8.4 0.8%, 8.2 0.2%, 8.3

## 6. Variantes descartadas

Las que se probaron antes de llegar al método actual. Las recoge
`docs/metricas/importancia-clausula-historia.md`.

### Prior por partes

Un PageRank personalizado por cada parte $a$, sobre el grafo entero, sembrado solo en sus celdas:
$\pi_a(v)=1/(|C_a|\cdot|V_{c,a}|)$ si $v\in V_{c,a}$. La importancia para la parte es
$I_a(c)=\sum_{v\in V_c}PR_a(v)$, y sin parte elegida se promedian las partes.

In [10]:
def full_graph(kg):
    keep = {n['id'] for c in ppr.NODE_COLLECTIONS for n in kg.get(c) or []}
    graph = nx.MultiGraph()
    graph.add_nodes_from(keep)
    for edge in kg['edges']:
        if edge['source'] in keep and edge['target'] in keep:
            graph.add_edge(edge['source'], edge['target'])
    return graph


def cells_of(kg):
    parties = {p['id'] for p in kg['parties']}
    cells = {}
    for collection in ppr.DEONTIC_COLLECTIONS:
        for statement in kg[collection]:
            party = owner(statement, collection)
            if statement.get('clauseId') and party in parties:
                cells.setdefault(party, {}).setdefault(statement['clauseId'], []).append(statement['id'])
    return cells


for path, kg, rows in contracts:
    graph = full_graph(kg)
    size = {c: len(g) for c, g in rows.items()}
    by_party = {}
    for party, cells in cells_of(kg).items():
        seed = {n: 0.0 for n in graph}
        for group in cells.values():
            for s in group:
                seed[s] = 1 / (len(cells) * len(group))
        rank = nx.pagerank(graph, alpha=ppr.DAMPING, personalization=seed, dangling=seed, tol=1e-13, max_iter=2000)
        by_party[party] = {c: sum(rank[s] for s in g) for c, g in rows.items()}
    average = {c: sum(i[c] for i in by_party.values()) / len(by_party) for c in rows}
    own_cell = [spearman(by_party[a], {c: len(cells_of(kg)[a].get(c, [])) for c in rows}) for a in by_party]
    between = f'{spearman(*by_party.values()):.2f}' if len(by_party) == 2 else f'({len(by_party)} partes)'
    print(f'{short(path):26} ρ promedio con el tamaño {spearman(average, size):>5.2f} · '
          f'ρ de I_a con su celda {min(own_cell):.2f} – {max(own_cell):.2f} · ρ entre partes {between}')

BELLICUMPHARMACEUTICALS_   ρ promedio con el tamaño  0.91 · ρ de I_a con su celda 0.89 – 0.90 · ρ entre partes 0.79


CARTAO_DE_CREDITO_SANTAN   ρ promedio con el tamaño  0.87 · ρ de I_a con su celda 0.90 – 0.93 · ρ entre partes 0.00
SteelVaultCorp_20081224_   ρ promedio con el tamaño  0.85 · ρ de I_a con su celda 0.86 – 0.93 · ρ entre partes 0.56
Synthetic_HalversonCorvi   ρ promedio con el tamaño  0.60 · ρ de I_a con su celda 0.39 – 0.95 · ρ entre partes (3 partes)
Synthetic_OrvaneTalvera_   ρ promedio con el tamaño  0.82 · ρ de I_a con su celda 0.82 – 0.87 · ρ entre partes -0.20
lease_contract_10          ρ promedio con el tamaño  0.82 · ρ de I_a con su celda 0.83 – 0.94 · ρ entre partes 0.46
lease_contract_166         ρ promedio con el tamaño  0.95 · ρ de I_a con su celda 0.90 – 0.96 · ρ entre partes 0.39
lease_contract_201         ρ promedio con el tamaño  0.89 · ρ de I_a con su celda 0.87 – 0.93 · ρ entre partes 0.93


### Entrada desde las demás cláusulas, con todo el grafo menos las partes

La forma actual de la importancia, pero con términos definidos, condiciones, referencias y
valores dentro del paseo. Fue la primera versión de la entrada que se programó.

In [11]:
without_parties = tuple(c for c in ppr.NODE_COLLECTIONS if c != 'parties')
for path, kg, rows in contracts:
    clauses, landed = landed_matrix(kg, rows, collections=without_parties)
    incoming = settled({c: (landed[i].sum() - landed[i, i]) / len(clauses) for i, c in enumerate(clauses)})
    size = {c: len(rows[c]) for c in clauses}
    zeros = sum(1 for v in incoming.values() if v == 0)
    rho = f'{spearman(incoming, size):.2f}' if zeros < len(rows) else '—'
    print(f'{short(path):26} ρ con el tamaño {rho:>5} · sin conexión {zeros:>2} de {len(rows)}')

BELLICUMPHARMACEUTICALS_   ρ con el tamaño  0.67 · sin conexión  0 de 91


CARTAO_DE_CREDITO_SANTAN   ρ con el tamaño  0.47 · sin conexión  2 de 81
SteelVaultCorp_20081224_   ρ con el tamaño -0.12 · sin conexión  6 de 22


Synthetic_HalversonCorvi   ρ con el tamaño -0.05 · sin conexión  2 de 19


Synthetic_OrvaneTalvera_   ρ con el tamaño  0.10 · sin conexión  9 de 48
lease_contract_10          ρ con el tamaño -0.12 · sin conexión  8 de 20


lease_contract_166         ρ con el tamaño     — · sin conexión  9 de 9
lease_contract_201         ρ con el tamaño -0.12 · sin conexión  2 de 8


### Partes como puente, con reparto justo

Cláusulas, enunciados y partes en el paseo, pero cada parte reparte lo que recibe primero a
partes iguales entre las filas donde aparece y después entre sus enunciados de esa fila.

In [12]:
for path, kg, rows in contracts:
    parties = {p['id'] for p in kg['parties']}
    keep = {n['id'] for c in ('parties', *ppr.INFLOW_COLLECTIONS) for n in kg.get(c) or []}
    clause_of = {s['id']: s.get('clauseId') for c in ppr.DEONTIC_COLLECTIONS for s in kg[c]}
    graph = nx.DiGraph()
    graph.add_nodes_from(keep)
    linked = {}
    for edge in kg['edges']:
        a, b = edge['source'], edge['target']
        if a not in keep or b not in keep:
            continue
        for x, y in ((a, b), (b, a)):
            if x in parties:
                linked.setdefault(x, set()).add(y)
            else:
                weight = graph[x][y]['weight'] + 1 if graph.has_edge(x, y) else 1
                graph.add_edge(x, y, weight=weight)
    for party, statements in linked.items():
        groups = {}
        for s in statements:
            groups.setdefault(clause_of.get(s) or s, []).append(s)
        for group in groups.values():
            for s in group:
                graph.add_edge(party, s, weight=1 / (len(groups) * len(group)))
    clauses = list(rows)
    landed = np.zeros((len(clauses), len(clauses)))
    for j, start in enumerate(clauses):
        seed = {n: 0.0 for n in graph}
        for s in rows[start]:
            seed[s] = 1 / len(rows[start])
        rank = nx.pagerank(graph, alpha=ppr.DAMPING, personalization=seed, dangling=seed, weight='weight', tol=1e-13, max_iter=2000)
        for i, clause in enumerate(clauses):
            landed[i, j] = sum(rank[s] for s in rows[clause])
    incoming = settled({c: (landed[i].sum() - landed[i, i]) / len(clauses) for i, c in enumerate(clauses)})
    size = {c: len(rows[c]) for c in clauses}
    values = sorted(incoming.values(), reverse=True)
    ties = sum(1 for x, y in zip(values, values[1:]) if x == y)
    zeros = sum(1 for v in incoming.values() if v == 0)
    print(f'{short(path):26} ρ con el tamaño {spearman(incoming, size):>5.2f} · sin conexión {zeros:>2} · empates {ties}')

BELLICUMPHARMACEUTICALS_   ρ con el tamaño  0.72 · sin conexión  0 · empates 3


CARTAO_DE_CREDITO_SANTAN   ρ con el tamaño  0.72 · sin conexión  0 · empates 11
SteelVaultCorp_20081224_   ρ con el tamaño  0.48 · sin conexión  0 · empates 4


Synthetic_HalversonCorvi   ρ con el tamaño  0.73 · sin conexión  1 · empates 2


Synthetic_OrvaneTalvera_   ρ con el tamaño  0.47 · sin conexión  0 · empates 17
lease_contract_10          ρ con el tamaño  0.55 · sin conexión  0 · empates 3
lease_contract_166         ρ con el tamaño  0.12 · sin conexión  0 · empates 4
lease_contract_201         ρ con el tamaño  0.54 · sin conexión  0 · empates 1
